# 2 · Corpus & speaker statistics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chemvatho/kolsch-tandem/blob/main/02_corpus/02_corpus_statistics.ipynb)

Quantify the digitised corpus: token/type counts, speaker demographics, and the
**IPA phoneme** and **orthography character** distributions — computed over the
transcript bodies from the registry.

## Setup

In [ ]:
!pip -q install pandas matplotlib
import pandas as pd, numpy as np, re, collections
import matplotlib.pyplot as plt

In [ ]:
# === Portable setup — identical paths in VS Code, Jupyter & Google Colab ===
import os, sys
from pathlib import Path
try:
    import google.colab  # noqa: F401
    _here = any((Path(p)/"kolsch_paths.py").exists() for p in [Path.cwd(), *Path.cwd().parents])
    if not _here and not Path("/content/kolsch-tandem/kolsch_paths.py").exists():
        os.system("git clone -q https://github.com/chemvatho/kolsch-tandem.git /content/kolsch-tandem")
except Exception:
    pass
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"kolsch_paths.py").exists()),
             Path("/content/kolsch-tandem"))
sys.path.insert(0, str(_root))
from kolsch_paths import ROOT, DATA, PAGES, AUDIO, TRANS, SEG, INDEX, LEXICON, MODELS
os.chdir(ROOT)
print("repo root:", ROOT)

In [ ]:
index = pd.read_csv(INDEX)

def read_body(path, drop_header=2):
    """Spoken text = transcript minus the header (speaker line) and title line."""
    lines = [l for l in open(path, encoding="utf-8").read().splitlines() if l.strip()]
    return " ".join(lines[drop_header:])

rows = []
for r in index.itertuples(index=False):
    rows.append({**r._asdict(), "body": read_body(os.path.join(DATA, r.transcript))})
df = pd.DataFrame(rows)                     # one row per recording
print(len(df), "recording(s) in corpus")
df[["id","speaker","age","occupation","neighbourhood","title"]]

## 1 · Token & type counts

In [ ]:
def tokens(text): return re.findall(r"\S+", str(text).lower())

all_tokens = [t for b in df["body"] for t in tokens(b)]
ttypes = set(all_tokens)
print(f"recordings  : {len(df)}")
print(f"word tokens : {len(all_tokens):,}")
print(f"word types  : {len(ttypes):,}")
print(f"type/token  : {len(ttypes)/max(1,len(all_tokens)):.3f}")
per_cd = df.assign(n=df['body'].map(lambda t: len(tokens(t)))).groupby('cd')['n'].sum()
print("\nwords per CD:\n", per_cd.to_string())

## 2 · Speaker demographics

In [ ]:
print("age range :", int(df.age.min()), "-", int(df.age.max()))
print("age mean  :", round(df.age.mean(),1), " median:", df.age.median())

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(df.age, bins=[10,20,30,40,50,60,70,80,90], color="#0F7C73", edgecolor="white")
ax[0].set_title("Speaker age distribution"); ax[0].set_xlabel("age"); ax[0].set_ylabel("speakers")
nb_counts = df.neighbourhood.value_counts().head(12)
ax[1].barh(nb_counts.index[::-1], nb_counts.values[::-1], color="#1F3864")
ax[1].set_title("Top neighbourhoods"); ax[1].set_xlabel("speakers")
fig.tight_layout(); plt.show()

## 3 · Distributions — IPA phonemes & orthography

Two distributions over the corpus: the **IPA phoneme** inventory (each word is
converted to IPA with the Kölsch G2P, `kolsch_g2p.py`, then split into phonemes)
and the **orthography character** inventory (the raw Kölsch spelling). This cell
re-derives the corpus from the registry if you didn't run the cells above.

In [ ]:
import collections, re
import matplotlib.pyplot as plt
from kolsch_g2p import word_to_ipa                       # Kölsch G2P (repo root)

# self-contained: re-derive the corpus from the registry if `df` isn't in memory
if "df" in globals() and "body" in getattr(df, "columns", []):
    bodies = list(df["body"].astype(str))
else:
    _idx = pd.read_csv(INDEX)
    def _read_body(p, drop=2):
        L = [l for l in open(p, encoding="utf-8").read().splitlines() if l.strip()]
        return " ".join(L[drop:])
    bodies = [_read_body(os.path.join(DATA, r.transcript)) for r in _idx.itertuples(index=False)]
print(len(bodies), "recording(s)")

# ---- IPA phoneme inventory + longest-match tokeniser (44 symbols) ----
PHONEMES = ['aː','ɛː','eː','iː','oː','uː','yː','øː','aɪ','aʊ','ɔɪ','ɛɪ','ɔʏ','ɐʊ',
 't͡s','p͡f','t͡ʃ','a','ɛ','ɪ','ɔ','ʊ','ʏ','œ','ə','ɐ','e','o','i','u','y','ø',
 'ʃ','ʒ','ç','χ','x','f','v','s','z','h','p','b','t','d','k','ɡ','g','ʔ','m','n','ŋ',
 'l','ʁ','j','r','w','ɥ']
_P = sorted(PHONEMES, key=len, reverse=True)
def to_phonemes(w):
    out, i = [], 0
    while i < len(w):
        for p in _P:
            if w.startswith(p, i): out.append(p); i += len(p); break
        else: i += 1
    return out

def _norm_orth(s):
    s = str(s).lower(); s = re.sub(r"\d+", "", s)
    return re.sub(r"[^a-zäöüßçïëéèáà']", "", s)          # Kölsch letters + apostrophe

ipa_counts, char_counts = collections.Counter(), collections.Counter()
for b in bodies:
    for w in b.split():
        ipa_counts.update(to_phonemes(word_to_ipa(w)))  # orthography -> IPA -> phonemes
    char_counts.update(_norm_orth(b))                    # orthography characters

def dist_plot(counts, xlabel, title, color, fname):
    keys = [k for k, _ in counts.most_common()]
    vals = [counts[k] for k in keys]
    fig, ax = plt.subplots(figsize=(16, 6))
    bars = ax.bar(range(len(keys)), vals, color=color, edgecolor="white")
    ax.bar_label(bars, padding=2, fontsize=8)
    ax.set_xticks(range(len(keys))); ax.set_xticklabels(keys, fontsize=13)
    ax.set_xlabel(xlabel); ax.set_ylabel("Count")
    ax.set_title(f"{title} — {len(keys)} units, {sum(vals):,} tokens")
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.25)
    fig.tight_layout(); plt.savefig(fname, dpi=150); plt.show()

dist_plot(ipa_counts,  "Kölsch phoneme (IPA)",          "Kölsch phoneme distribution (IPA)",
          "#0F7C73", "kolsch_ipa_distribution.png")
dist_plot(char_counts, "Kölsch orthographic character",  "Kölsch orthography distribution",
          "#1F3864", "kolsch_orthography_distribution.png")

## Summary

Headline corpus numbers (tokens, types, TTR, per-CD), speaker demographics, and
the two distributions (`kolsch_ipa_distribution.png`,
`kolsch_orthography_distribution.png`) that feed the data section of the paper.